<a href="https://colab.research.google.com/github/bhanukumar1729/CabinConnect/blob/main/Geometry_rad_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import Section

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# from sklearn.linear_model import LinearRegression
# from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
# from sklearn.svm import SVR
# import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


#data loading

In [ ]:

EXCEL_FILES = [
    "0.9 um.xlsx",
    "1.1 um.xlsx",
    "1.5 um.xlsx",
    "1.3  um.xlsx"
]
TEST_SIZE = 0.2
RANDOM_STATE = 42
FREQUENCY_COL = "Frequency"
TARGET_COL = "Transmission coefficient"
VALUE_COL_NAME = "Value"
all_dfs = []

for file in EXCEL_FILES:
    print(f"\nReading file: {file}")
    sheets_dict = pd.read_excel(file, sheet_name=None)
    for sheet_name, df in sheets_dict.items():
        try:
            value = float(sheet_name[0:3])
        except ValueError:
            print(f"Skipping sheet '{sheet_name}' (not numeric)")
            continue
        df = df.copy()
        df[VALUE_COL_NAME] = value
        all_dfs.append(df)
        print(f"  Loaded sheet {sheet_name} → {len(df)} rows")
FEATURE_COLUMNS = ["Frequency", "Value"]
TARGET_COLUMN = "Transmission coefficient"
full_df = pd.concat(all_dfs, ignore_index=True)
print("\nFinal Dataset Shape:", full_df.shape)
print(full_df.head())



Reading file: 0.9 um.xlsx


FileNotFoundError: [Errno 2] No such file or directory: '0.9 um.xlsx'

In [ ]:
print(full_df.tail())

#TRAIN–TEST SPLIT + SCALING

In [ ]:
X = full_df[FEATURE_COLUMNS]
y = full_df[TARGET_COLUMN]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)


#METRICS FUNCTION

In [ ]:
def compute_metrics(y_true, y_pred, train_time, pred_time):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Efficiency (%)": r2 * 100,
        "Train Time (s)": train_time,
        "Prediction Time (s)": pred_time
    }


#Train model

In [ ]:
def train_eval_model(name, model, X_train, y_train, X_test, y_test):
    print(f"Training {name}")

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start

    return compute_metrics(y_test, y_pred, train_time, pred_time), y_pred


#NEURAL NETWORK TRAINING

In [ ]:
def train_eval_nn(X_train, y_train, X_test, y_test):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(64, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')

    early_stop = EarlyStopping(patience=20, restore_best_weights=True)

    start = time.time()
    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test).flatten()
    pred_time = time.time() - start

    return compute_metrics(y_test, y_pred, train_time, pred_time), y_pred


#Models

In [ ]:
models = {
    # "Linear Regression": LinearRegression(),
    # "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    # "XGBoost": xgb.XGBRegressor(objective="reg:squarederror", n_estimators=100),
    # "SVR": SVR(kernel="rbf", C=100, epsilon=0.01)
}


#Running Models

In [ ]:
results = {}
predictions = {}

for name, model in models.items():
    metrics, y_pred = train_eval_model(
        name, model, X_train, y_train, X_test, y_test
    )
    results[name] = metrics
    predictions[name] = y_pred

# nn_metrics, nn_pred = train_eval_nn(X_train, y_train, X_test, y_test)
# results["Neural Network"] = nn_metrics
# predictions["Neural Network"] = nn_pred


#Results table

In [ ]:
results_df = pd.DataFrame(results).T.sort_values(by="R2", ascending=False)
display(results_df)

BEST_MODEL_NAME = results_df.index[0]
BEST_Y_PRED = predictions[BEST_MODEL_NAME]

print("Best Model:", BEST_MODEL_NAME)


#Graph


In [ ]:
def plot_best_model_line_graph(
    X_test_raw,
    y_test,
    y_pred_best,
    value_to_plot,
    value_column,              # <-- dynamic column
    best_model_name,
    target_col="Transmission coefficient"
):
    plot_df = X_test_raw.copy()
    plot_df["Actual"] = y_test.values
    plot_df["Predicted"] = y_pred_best

    if value_column not in plot_df.columns:
        raise ValueError(
            f"Column '{value_column}' not found. "
            f"Available columns: {list(plot_df.columns)}"
        )

    subset = plot_df[plot_df[value_column] == value_to_plot] \
        .sort_values(by="Frequency")

    if subset.empty:
        print(f"No test data found for {value_column} = {value_to_plot}")
        return

    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 5))

    plt.plot(
        subset["Frequency"],
        subset["Actual"],
        label="Actual",
        linewidth=2
    )

    plt.plot(
        subset["Frequency"],
        subset["Predicted"],
        linestyle="--",
        label=f"Predicted ({best_model_name})",
        linewidth=2
    )

    plt.xlabel("Frequency")
    plt.ylabel(target_col)
    plt.title(
        f"{target_col} vs Frequency\n"
        f"Best Model: {best_model_name}, {value_column} = {value_to_plot}"
    )
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.show()


In [ ]:
plot_best_model_line_graph(
    X_test_raw=X_test_raw,
    y_test=y_test,
    y_pred_best=BEST_Y_PRED,
    value_to_plot=1.3,
    value_column="Value",
    best_model_name=BEST_MODEL_NAME
)


In [ ]:
value_column = [c for c in X_test_raw.columns if c != "Frequency"][0]
print("Using value column:", value_column)


#HeatMaps
